# Rough Implementation of Blueprint
 This notebook contains a rough implementation of the Blueprint. It is used to:
 - **Verify** if the concept works in reality
 - **Brainstorm** different **approaches** for each solution
- Identify **edge cases** in the solution.

Here in our demo we will use the **"Qwen 3.5 9B Q4 K M" (in GGUF format)** Open Source model as the central control brain.
And to use that LLM we will use **Llama.cpp (Python Library)**.

In [2]:
from llama_cpp import Llama, LlamaGrammar
# Llama grammar is required for forcing the output to be in our expected format.

We are **simulating** the industry server on our local computer.

Since on my **Mac** model is located in the downloads folder `/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf`

For real work it needs alot more context to breath, but for testing purpose I am keeping it small `2000`, so that it consumes less RAM.

In [3]:
qwen = Llama(
    model_path = "/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf",
    n_ctx = 2000,
    stream = True,
    verbose = True # For agentic task we don't need verbose responses
)
print("Model Loaded Successfully!!")

output = qwen.create_chat_completion(
    messages=[{"role": "system", "content": "Hello How can I help you?"},
              {"role": "user", "content": "Hello"}],
    temperature=0,
    max_tokens=1000,
)
print(output)

ggml_metal_device_init: testing tensor API for f16 support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x103195bc0 | th_max = 1024 | th_width =   32
ggml_metal_device_init: testing tensor API for bfloat support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x103195b80 | th_max = 1024 | th_width =   32
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 0.024 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0 (Apple M5)
ggml_metal_device_init: GPU family: MTLGPUFamilyApple10  (1010)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFam

Model Loaded Successfully!!


llama_perf_context_print:        load time =    2401.58 ms
llama_perf_context_print: prompt eval time =    2383.51 ms /    25 tokens (   95.34 ms per token,    10.49 tokens per second)
llama_perf_context_print:        eval time =    5016.72 ms /    28 runs   (  179.17 ms per token,     5.58 tokens per second)
llama_perf_context_print:       total time =    7434.74 ms /    53 tokens
llama_perf_context_print:    graphs reused =         27


{'id': 'chatcmpl-2a43ab39-f0b2-4833-80b1-2b5ca81b67b5', 'object': 'chat.completion', 'created': 1790266170, 'model': '/Users/noyan/downloads/Qwen3.5-9B-Q4_K_M.gguf', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Hello! How can I help you today? Feel free to ask me any questions or let me know if you need assistance with anything specific.'}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 25, 'completion_tokens': 28, 'total_tokens': 53}}


In [3]:
completion = output["choices"][0]["message"]["content"]
print(completion)

Hello! How can I help you today? Feel free to ask me any questions or let me know if you need assistance with anything specific.


For now, we first need to brainstorm on the capabilities of the agent and design the execution for each of the task.
Capabilities:
1. Read/Write Files
2. List files, can create directories
3. Fetch the metadata for each file.
4. Managing git like log and version control, where each commit is trackable.
5. Run command in the sandbox, code execution (Python or JavaScript or both)

We will store the relative path of the workspace of the agent in the `rel_path` variable.
We will use `pathlib` to deal with paths.

- `.resolve()` creates absolute path from relative path, with respect to working directory
- `Path()` converts the input into a path variable.
- `.mkdir(exist_ok = True, parents = True)` It creates directory at the given location, `exist_ok = True` ensures no error occurs if the directory already exists, `parents = True` creates any parent directory if missing instead of throwing errors.

In [6]:
from pathlib import Path
rel_path = Path('./workspace').resolve()
print(rel_path)
rel_path.mkdir(exist_ok = True, parents = True)

/Users/noyan/Desktop/Smart India Hackathon/AgenticAI/Blueprint/workspace


Since we are simulating the project here, I will create a directory called `LocalStorage`, which behaves as the local data base of the industry.

The **central_backend.py** will act as a **Backbone** for the project.

The **sandbox.py** will help in code execution by **LLM**.

The **Interpreter** directory contains interpreters for different types of files.

The **LLM** directory will contain the open-source models in gguf format.

The **cli.py** will be the Command Line Interface Application to use the agent.

The **GUI** directory will be the Graphical User Interface Application to use the Agent.

Since **Central Backend** is the _spinal cord_ of this project, which controls the entire data flow in the project, it needs to be build first.

The core algorithm behind **Central Backend** is:
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt.
3. Proceed for task created by **LLM**.
4. **Feed** the output of task, along with the previous context to the LLM.
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

**Github Issue 1:** _Chat Completion method of Agents_
Solution Algorithm (Simplified Version):
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt and perform reasoning task on user's input and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
3. Get the **JSON** sequence of tasks and Proceed for those tasks **simultaneously**.
4. **Feed** the output of task, along with the previous context to the LLM and perform reasoning task on context and output and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

**Github Issue 3:** _Inefficient file search by Central Backend_
Solution Algorithm (Simplified Version):
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt and perform reasoning task on user's input and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
3. Get the **JSON** sequence of tasks and Proceed for those tasks **simultaneously**, and for file reading tasks to get just the context of each file, make use of LLM prepared one-time meta-data.
4. **Feed** the output of task, along with the previous context to the LLM and perform reasoning task on context and output and identify which tasks are independent and can be done simultaneously to get the idea of the workspace.
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

**Github Issue 2:** _One way prompt to generation_
Solution Algorithm (Simplified Version):
1. Take **User's Input**.
2. **Feed** it to LLM with the prompt and perform reasoning task on user's input and identify which tasks are independent and can be done simultaneously to get the idea of the workspace (spit out the tokens being generated for reasoning).
3. Get the **JSON** sequence of tasks and Proceed for those tasks **simultaneously**, and for file reading tasks to get just the context of each file, make use of LLM prepared one-time meta-data. (Give the user an opportunity to talk to the agent live, while LLM gives out some reasoning along with tasks, Nudge the user's query into the context by safely maintaing the correct structure of the context)
4. **Feed** the output of task, along with the previous context to the LLM and perform reasoning task on context and output and identify which tasks are independent and can be done simultaneously to get the idea of the workspace. (Show the reasoning and give the user, a chance again)
5. Get the new tasks and **repeat** from step 3, until all tasks are finished.
6. Show the **final inference** of the session.

# 🎉 Congratulations!! We are done with the **algorithmic part**, and we just need to translate it to Python code by handeling edge cases.

In [4]:
# Testing for streaming the response
stream = qwen.create_chat_completion(
    messages=[{"role": "system", "content": "think for every response"}, {"role": "user", "content": "Hello, who are you?"}],
    stream=True, # To enable streaming
    max_tokens=512,
)

for chunk in stream:
    delta = chunk["choices"][0]["delta"]
    if "content" in delta:
        print(delta["content"], end="", flush=True)
print(stream)  # final newline

Llama.generate: 3 prefix-match found but partial kv removal not supported, re-evaluating full prompt


Hello! I am Qwen3.5, the latest large language model developed by Alibaba Cloud. I'm designed to assist with a wide range of tasks, from answering questions and creating content to analyzing data and coding. How can I help you today? 😊

llama_perf_context_print:        load time =    2401.58 ms
llama_perf_context_print: prompt eval time =    1825.14 ms /    27 tokens (   67.60 ms per token,    14.79 tokens per second)
llama_perf_context_print:        eval time =    7691.08 ms /    53 runs   (  145.11 ms per token,     6.89 tokens per second)
llama_perf_context_print:       total time =    9599.77 ms /    80 tokens
llama_perf_context_print:    graphs reused =         52


<generator object _convert_text_completion_chunks_to_chat at 0x10b48ec40>


# ✅ Now we have everything for Implementation and we can proceed with the development of our final python code.